# 01 - Data Prep and EDA

Objetivo:
- crear splits reproducibles
- validar distribución por clase
- inspeccionar ejemplos

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from real_estate_ml.config import load_config
from real_estate_ml.constants import CLASSES
from real_estate_ml.data.prepare_splits import prepare_splits

cfg = load_config(ROOT / "configs" / "base_config.yaml")
cfg

In [ ]:
prepare_splits(
    raw_training_dir=ROOT / cfg["data"]["raw_training_dir"],
    raw_validation_dir=ROOT / cfg["data"]["raw_validation_dir"],
    output_dir=ROOT / cfg["data"]["data_dir"],
    train_split=cfg["data"]["train_split"],
    val_split=cfg["data"]["val_split"],
    test_split=cfg["data"]["test_split"],
    seed=cfg["training"]["seed"],
)

In [ ]:
def count_images(split_name: str):
    rows = []
    split_dir = ROOT / cfg["data"]["data_dir"] / split_name
    for class_name in CLASSES:
        class_dir = split_dir / class_name
        count = len(list(class_dir.glob("*.jpg"))) + len(list(class_dir.glob("*.jpeg"))) + len(list(class_dir.glob("*.png")))
        rows.append({"split": split_name, "class": class_name, "count": count})
    return pd.DataFrame(rows)

df = pd.concat([count_images("train"), count_images("val"), count_images("test")], ignore_index=True)
df.head()

In [ ]:
pivot = df.pivot(index="class", columns="split", values="count")
pivot.plot(kind="bar", figsize=(14, 6), rot=45)
plt.title("Images per class and split")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

pivot